In [ ]:
from difflib import SequenceMatcher
from pyspark.sql.functions import col
import builtins

# Función auxiliar para calcular similitud (0 a 1)
def string_similarity(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

print("\n" + "="*70)
print("PASO 4: DETALLES DE LAS REGLAS RELEVANTES (FILTRANDO SECUELAS)")
print("="*70)

# ==============================
# 1. Convertir reglas a lista (un poco más de 10 para tener margen tras filtrar)
# ==============================
top_rules_list = rules_filtered.sortWithinPartitions(col("lift").desc()).limit(50).collect()

# ==============================
# 2. Diccionario movieId → title (ya lo tienes)
# ==============================
movie_dict = {
    int(row["movieId"]): row["title"]
    for row in movies.select("movieId", "title").collect()
}

# ==============================
# 3. Mostrar reglas filtrando similitud > 90%
# ==============================
rules_shown = 0
MAX_RULES_TO_DISPLAY = 10
SIMILARITY_THRESHOLD = 0.85 # 85-90% suele detectar secuelas (Toy Story 1 vs Toy Story 2)

for row in top_rules_list:
    if rules_shown >= MAX_RULES_TO_DISPLAY:
        break

    antecedent_ids = row["antecedent"]
    consequent_ids = row["consequent"]

    # Obtener títulos
    antecedent_titles = [movie_dict.get(int(i), f"ID:{i}") for i in antecedent_ids]
    consequent_titles = [movie_dict.get(int(i), f"ID:{i}") for i in consequent_ids]

    # --- LÓGICA DE FILTRADO DE SIMILITUD ---
    is_too_similar = False
    for a_title in antecedent_titles:
        for c_title in consequent_titles:
            # Si el título es 90% igual, lo marcamos como secuela/obvio
            if string_similarity(a_title, c_title) > SIMILARITY_THRESHOLD:
                is_too_similar = True
                break
        if is_too_similar: break

    if is_too_similar:
        continue # Saltamos esta regla y vamos a la siguiente
    # ---------------------------------------

    rules_shown += 1
    print(f"\n╔═ REGLA {rules_shown} ═{'='*65}")
    print(f"║")
    print(f"║   SI usuario calificó positivamente:")
    for title in antecedent_titles:
        print(f"║      → {title}")
    print(f"║")
    print(f"║   ENTONCES es probable que también calificó positivamente:")
    for title in consequent_titles:
        print(f"║      → {title}")
    print(f"║")
    print(f"║   Métricas:")
    print(f"║      • Soporte:   {row['support']:.4f}")
    print(f"║      • Confianza: {row['confidence']:.4f}")
    print(f"║      • Lift:      {row['lift']:.4f}x")

    # Interpretación rápida
    strength = "MUY FUERTE" if row['lift'] > 1.5 else "FUERTE"
    print(f"║      • Fuerza:    {strength}")
    print(f"╚{'='*71}")

if rules_shown == 0:
    print("No se encontraron reglas que superen los filtros de relevancia y diversidad.")

In [ ]:
from difflib import SequenceMatcher
from pyspark.sql.functions import col

# ==========================================
# Función similitud títulos
# ==========================================
def string_similarity(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

print("\n" + "="*70)
print("TOP 5 REGLAS CON MENOR LIFT (RELACIONES DÉBILES)")
print("="*70)

# ==========================================
# Obtener reglas con MENOR lift
# ==========================================
low_lift_rules = (
    rules_filtered
    .filter(col("lift") > 0)  # evitar lift inválido
    .sortWithinPartitions(col("lift").asc())
    .limit(50)
    .collect()
)

# ==========================================
# Diccionario movieId → title
# ==========================================
movie_dict = {
    int(row["movieId"]): row["title"]
    for row in movies.select("movieId", "title").collect()
}

# ==========================================
# Mostrar reglas
# ==========================================
rules_shown = 0
MAX_RULES_TO_DISPLAY = 5
SIMILARITY_THRESHOLD = 0.3

for row in low_lift_rules:

    if rules_shown >= MAX_RULES_TO_DISPLAY:
        break

    antecedent_titles = [
        movie_dict.get(int(i), f"ID:{i}")
        for i in row["antecedent"]
    ]

    consequent_titles = [
        movie_dict.get(int(i), f"ID:{i}")
        for i in row["consequent"]
    ]

    # ==========================================
    # Filtrar secuelas / títulos casi iguales
    # ==========================================
    is_too_similar = False

    for a_title in antecedent_titles:
        for c_title in consequent_titles:

            if string_similarity(a_title, c_title) > SIMILARITY_THRESHOLD:
                is_too_similar = True
                break

        if is_too_similar:
            break

    if is_too_similar:
        continue

    # ==========================================
    # Mostrar regla
    # ==========================================
    rules_shown += 1

    print(f"\n╔═ REGLA DÉBIL {rules_shown} ═{'='*57}")
    print(f"║")
    print(f"║   SI usuario calificó positivamente:")

    for title in antecedent_titles:
        print(f"║      → {title}")

    print(f"║")
    print(f"║   ENTONCES también podría haber visto:")

    for title in consequent_titles:
        print(f"║      → {title}")

    print(f"║")
    print(f"║   Métricas:")
    print(f"║      • Soporte:   {row['support']:.4f}")
    print(f"║      • Confianza: {row['confidence']:.4f}")
    print(f"║      • Lift:      {row['lift']:.4f}x")

    # ==========================================
    # Interpretación
    # ==========================================
    if row['lift'] < 1:
        strength = "RELACIÓN NEGATIVA"
    elif row['lift'] < 1.2:
        strength = "CASI INDEPENDIENTE"
    else:
        strength = "DÉBIL"

    print(f"║      • Interpretación: {strength}")

    print(f"╚{'='*71}")

if rules_shown == 0:
    print("No se encontraron reglas débiles válidas.")

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from difflib import SequenceMatcher
from pyspark.sql.functions import col

# 1. Función de similitud (definirla aquí asegura que exista)
def string_similarity(a, b):
    return SequenceMatcher(None, str(a).lower(), str(b).lower()).ratio()

# 2. Aseguramos que rules_filtered exista y tenga los datos frescos
# Usamos rules_enriched que ya definiste en tu query anterior
rules_filtered = rules_enriched.filter(
    (col("confidence") >= 0.2) &
    (col("lift") > 1.2)  # Subimos un poco el lift para mejores asociaciones
)

# 3. Recolectamos un poco más de 10 para tener margen tras el filtro de similitud
# .collect() es seguro aquí porque solo traemos 30 filas a la memoria local
raw_top_rules = rules_filtered.sortWithinPartitions(col("lift").desc()).limit(30).collect()

# 4. Construcción del Grafo
G = nx.DiGraph()
SIMILARITY_THRESHOLD = 0.85
rules_added = 0

for row in raw_top_rules:
    if rules_added >= 10: break # Límite estricto pedido

    u_id, v_id = row["antecedent"][0], row["consequent"][0]
    u_title = movie_dict.get(int(u_id), f"ID:{u_id}")
    v_title = movie_dict.get(int(v_id), f"ID:{v_id}")

    if string_similarity(u_title, v_title) < SIMILARITY_THRESHOLD:
        G.add_edge(u_title, v_title, weight=row["lift"])
        rules_added += 1

# 5. Renderizado del Grafo
plt.figure(figsize=(14, 9))
pos = nx.spring_layout(G, k=1.2, seed=42) # k controla la distancia entre nodos

# Dibujamos nodos
nx.draw_networkx_nodes(G, pos, node_size=3500, node_color='#5dade2',
                       edgecolors='white', linewidths=2, alpha=0.9)

# Dibujamos flechas (curvas para que se vea Pro)
nx.draw_networkx_edges(G, pos, width=2, edge_color='#abb2b9',
                       arrowsize=20, connectionstyle='arc3,rad=0.15')

# Dibujamos etiquetas
nx.draw_networkx_labels(G, pos, font_size=10, font_family='sans-serif', font_weight='bold')

# Etiquetas de Lift en las flechas
edge_labels = {(u, v): f"{G[u][v]['weight']:.2f}x" for u, v in G.edges()}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=9, font_color='red')

plt.title("Grafo de Recomendaciones: Top 10 Asociaciones No Obvias",
          fontsize=15, fontweight='bold', pad=20)
plt.axis('off')
plt.show()